# Segmentación de Clientes con K-Means: Costos Médicos

Este notebook identifica perfiles de riesgo en la cartera de asegurados mediante clustering no supervisado (K-Means), usando las variables `age`, `bmi` y `smoker`.

## Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

IMG_PATH = Path("../../images")
IMG_PATH.mkdir(exist_ok=True)

MODELS_PATH = Path("../../models")
MODELS_PATH.mkdir(exist_ok=True)

## 1. Carga y preparación de datos

Se carga el dataset `insurance.csv` y se crean dos variables binarias derivadas:
- `smoker`: 1 si el asegurado fuma, 0 en caso contrario.
- `obese`: 1 si el BMI es ≥ 30, 0 en caso contrario.

Finalmente se seleccionan las variables `bmi`, `age` y `smoker` para el clustering.

In [2]:
df_path = Path("../../data/insurance.csv")
df_insurance = pd.read_csv(df_path)
df_insurance['smoker'] = (df_insurance['smoker'] == "yes").astype(int)
df_insurance['obese'] = (df_insurance['bmi'] >= 30).astype(int)
df_insurance_filter = df_insurance[['bmi','age','smoker']]
print(df_insurance_filter)

         bmi  age  smoker
0     27.900   19       1
1     33.770   18       0
2     33.000   28       0
3     22.705   33       0
4     28.880   32       0
...      ...  ...     ...
1333  30.970   50       0
1334  31.920   18       0
1335  36.850   18       0
1336  25.800   21       0
1337  29.070   61       1

[1338 rows x 3 columns]


## 2. Estandarización de variables

K-Means es sensible a la escala de las variables. Se aplica `StandardScaler` para que cada variable tenga media 0 y desviación estándar 1, evitando que `bmi` o `age` dominen la distancia euclidiana.

In [3]:
## Creacion del objeto Scaler
scaler = StandardScaler()
## Estandarizacion del df, fit obtiene la media y la desviacion (distancia entre puntos) y transformo los estandariza
df_scaled = scaler.fit_transform(df_insurance_filter)
print(df_scaled[:5])

[[-0.45332    -1.43876426  1.97058663]
 [ 0.5096211  -1.50996545 -0.5074631 ]
 [ 0.38330685 -0.79795355 -0.5074631 ]
 [-1.30553108 -0.4419476  -0.5074631 ]
 [-0.29255641 -0.51314879 -0.5074631 ]]


## 3. Determinación del número óptimo de clusters con Elbow Method

Se entrena K-Means para k=1 hasta k=10 y se grafica el WCSS (Within-Cluster Sum of Squares) por cada valor de k. El punto donde la curva deja de descender bruscamente indica el número óptimo de clusters.

In [ ]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 4))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method: Número óptimo de clusters')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('WCSS: Dispersión interna')
plt.savefig(IMG_PATH / "07_elbow_method.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Entrenamiento del modelo y asignación de segmentos

Con k=3 seleccionado por el método del codo, se entrena el modelo final y se asigna una etiqueta de segmento a cada registro del dataset.

In [5]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(df_scaled)
df_insurance['segmento_riesgo'] = kmeans.labels_
print(df_insurance['segmento_riesgo'].value_counts().sort_index())

joblib.dump(scaler, MODELS_PATH / "standard_scaler.pkl")
joblib.dump(kmeans, MODELS_PATH / "kmeans_risk_segments.pkl")
print("Modelos guardados en models/")

segmento_riesgo
0    516
1    548
2    274
Name: count, dtype: int64
Modelos guardados en models/


## 5. Perfil promedio por segmento

Se calcula la media de `age`, `bmi`, `smoker` y `charges` por segmento para interpretar el perfil de riesgo de cada grupo.

In [6]:
df_insurance.groupby('segmento_riesgo')[['age', 'bmi', 'smoker', 'charges']].mean().round(2)

,age,bmi,smoker,charges
segmento_riesgo,,,,
0,26.81,29.43,0.0,5059.76
1,51.22,31.80,0.0,11611.73
2,38.51,30.71,1.0,32050.23


## 6. Visualización de segmentos: BMI vs Charges

Gráfico de dispersión que muestra la separación entre los tres segmentos de riesgo en el espacio BMI / costo médico.

In [ ]:
plt.figure(figsize=(10, 4))
sns.scatterplot(data=df_insurance, x='bmi', y='charges', 
                hue='segmento_riesgo', palette='Set1', alpha=0.6)
plt.title('Segmentos de Riesgo: BMI vs Charges')
plt.xlabel('BMI')
plt.ylabel('Charges (USD)')
plt.savefig(IMG_PATH / "08_risk_segments_scatter.png", dpi=150, bbox_inches='tight')
plt.show()